# Training Notebook

## 1. Setup & Environment

### 1.1 Mount Drive

Mount Google Drive for data and checkpoint access.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### 1.2 Import Libraries

Import libraries grouped by purpose: file/data handling, numerical/tabular, PyTorch core, computer vision, experiment tracking, split utilities and visualisation.

In [ ]:
# File handling
import os
import random
import shutil
import zipfile
from pathlib import Path
import json

# Numerical / tabular
import numpy as np
import pandas as pd

# PyTorch core
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

# Computer vision
import torchvision
from torchvision import models, transforms
from PIL import Image

# Experiment tracking
import wandb

# Split utilities (stratified split + stratified k-fold)
from sklearn.model_selection import train_test_split, StratifiedKFold

# Visualisation
import matplotlib.pyplot as plt

In [ ]:
# Version check
print(f'torch: {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'wandb: {wandb.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
  print(f'GPU: {torch.cuda.get_device_name(0)}')

### 1.3 Reproducibility

Two independent sources of randomness:


*   data-split randomness
*   training randomness

Both are pinned to one global seed, logged into every WandB run's config for traceability.

Bit-exact GPU reproducibility was decided as unnecessary. Therefore, cuDNN flag `deterministic` is set to `False`.

Considering every image is fixed at 224 × 224, cuDNN flag `benchmark` is set to `True` to allow cuDNN to auto-tune once and reuse the fastest algorithm for the rest of the run, at no real cost.

In [ ]:
SEED = 42

def seed_everything(seed=SEED):
  """
  Seeds every RNG source PyTorch code can touch. Called once at notebook start
  for global reproducibility, and again inside each sweep trial / training run
  so results don't depend on call order or prior runs.
  """
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

  torch.backends.cudnn.deterministic = False
  torch.backends.cudnn.benchmark = True

seed_everything()
print(f'Global seed set: {SEED}')
print(f'cudnn.deterministic: {torch.backends.cudnn.deterministic}')
print(f'cudnn.benchmark: {torch.backends.cudnn.benchmark}')

### 1.4 Experiment Tracking: Weights & Biases Setup

Authenticate via Colab Secrets. Defines the project constant and a helper function that builds consistent run identifiers (group, job_type, tags, name) across every run type in this notebook.

In [ ]:
from google.colab import userdata

WANDB_API_KEY = userdata.get('WANDB_API_KEY')
wandb.login(key=WANDB_API_KEY)
os.environ['WANDB_SILENT'] = 'true'

WANDB_ENTITY = 'thesis-mine'
WANDB_PROJECT = 'breast-ultrasound-classification'

In [ ]:
def build_run_identity(architecture, mode, arm, job_type, fold=None, eval_data=None):
  """
  Builds a consistent (group, job_type, tags, name) tuple for any run in this
  notebook, so every sweep trial, CV fold, retrain, and eval run is filterable
  and human-readable in the WandB UI later.

  architecture : 'resnet50' | 'efficientnet_b0' | 'swin_tiny'
  mode         : 'TL' | 'scratch'
  arm          : 'annotated' | 'inpainted' | 'masked'
  job_type     : 'sweep' | 'cv_fold' | 'final_retrain' | 'eval'
  fold         : int, only for job_type='cv_fold'
  eval_data    : 'busi_test' | 'bus_uclm' | 'breast', only for job_type='eval'
  """
  group = f'{architecture}_{mode}_{arm}'
  tags = [f'arch:{architecture}', f'mode:{mode}', f'arm:{arm}']

  name = group
  if job_type =='cv_fold':
    if fold is None:
      raise ValueError("job_type='cv_fold' requires a fold number.")
    tags.append(f'fold:{fold}')
    name += f'_fold{fold}'
  elif job_type == 'eval':
    if eval_data is None:
      raise ValueError("job_type='eval' requires eval_data.")
    tags.append(f'eval_data:{eval_data}')
    name += f'_eval_{eval_data}'
  else:
    name += f'_{job_type}'

  return {'group': group, 'job_type': job_type, 'tags': tags, 'name': name}

In [ ]:
# Sanity check: confirm the identity scheme produces the expected result

print(build_run_identity('resnet50', 'TL', 'inpainted', job_type='sweep'))
print(build_run_identity('resnet50', 'TL', 'inpainted', job_type='cv_fold', fold=3))
print(build_run_identity('resnet50', 'TL', 'inpainted', job_type='eval', eval_data='bus_uclm'))

### 1.5 Data Staging

Copies the final, processed dataset from Drive to local Colab storage to avoid I/O bottlenecks during training. Zips on Drive if not already zipped, copies the zip locally, unzips, then verifies file counts against Notebook 02's confirmed final structure.

*Note*: If the data changes upstream (e.g., Notebook 02), the zip file has to be manually deleted and recreated in order to reflect the upstream changes.

In [ ]:
DRIVE_FINAL_DIR = Path('/content/drive/MyDrive/University/Masters/MSc_Thesis/data/final')
DRIVE_ZIP_PATH = DRIVE_FINAL_DIR.parent / 'final.zip'

LOCAL_DATA_DIR = Path('/content/data')
LOCAL_ZIP_PATH = Path('/content/final.zip')

print(f'Drive source: {DRIVE_FINAL_DIR}')
print(f'Drive source exists: {DRIVE_FINAL_DIR.exists()}')

#### Zip on Drive if needed

In [ ]:
if DRIVE_ZIP_PATH.exists():
  print(f'Zip already exists on Drive: {DRIVE_ZIP_PATH}')
  print(f'Size: {DRIVE_ZIP_PATH.stat().st_size / (1024**3):.2f} GB')
else:
  print('No existing zip found -- creating one now.')

  archive_base = str(DRIVE_ZIP_PATH.with_suffix(''))
  shutil.make_archive(archive_base, 'zip', root_dir=DRIVE_FINAL_DIR)

  print(f'Zip created: {DRIVE_ZIP_PATH}')
  print(f'Size: {DRIVE_ZIP_PATH.stat().st_size / (1024**3):.2f} GB')

#### Local copy and unzip

In [ ]:
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
print('Copied to local storage.')

with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zf:
  zf.extractall(LOCAL_DATA_DIR)
print('Extracted successfully.')

print(f'Local data directory: {LOCAL_DATA_DIR}')

#### Verify Staged Data

Counts images per dataset / arm / class in the extracted local copy and checks against Notebook 02's confirmed final counts. Stops the notebook here if anything doesn't match, rather than proceeding on unverified data.

In [ ]:
# Confirmed final counts, from Notebook 02.
EXPECTED_COUNTS = {
    ('BUSI/BUSI_annotated', 'benign'): 296, ('BUSI/BUSI_annotated', 'malignant'): 161, ('BUSI/BUSI_annotated', 'normal'): 73,
    ('BUSI/BUSI_inpainted', 'benign'): 296, ('BUSI/BUSI_inpainted', 'malignant'): 161, ('BUSI/BUSI_inpainted', 'normal'): 73,
    ('BUSI/BUSI_masked', 'benign'): 296, ('BUSI/BUSI_masked', 'malignant'): 161, ('BUSI/BUSI_masked', 'normal'): 73,
    ('BUS-UCLM', 'benign'): 154, ('BUS-UCLM', 'malignant'): 71, ('BUS-UCLM', 'normal'): 404,
    ('BrEaST', 'benign'): 154, ('BrEaST', 'malignant'): 98, ('BrEaST', 'normal'): 4,
}

def scan_local_counts(base_dir):
  """
  Walks the extracted structure and counts image files per (dataset/arm, class) folder.
  Returns a dict keyed identically to EXPECTED_COUNTS, for direct comparison.
  """
  counts = {}
  for (subpath, cls) in EXPECTED_COUNTS.keys():
    class_dir = base_dir / subpath / cls
    if not class_dir.exists():
      counts[(subpath, cls)] = None
    else:
      counts[(subpath, cls)] = len(list(class_dir.glob('*.png')))
  return counts

actual_counts = scan_local_counts(LOCAL_DATA_DIR)

In [ ]:
mismatches = []

for key, expected in EXPECTED_COUNTS.items():
  actual = actual_counts[key]
  status = 'OK' if actual == expected else 'MISMATCH'
  if status == 'MISMATCH':
    mismatches.append((key, expected, actual))
  print(f'{key}: expected {expected}, found {actual} -- {status}')

total_expected = sum(EXPECTED_COUNTS.values())
total_actual = sum(v for v in actual_counts.values() if v is not None)
print(f'\nTotal expected: {total_expected}')
print(f'\nTotal found: {total_actual}')

if mismatches:
  print(f'\n{len(mismatches)} mismatch(es) found:')
  for key, expected, actual in mismatches:
    print(f'  {key}: expected {expected}, got {actual}')
  raise ValueError('Staged data does not match Notebook 02 confirmed counts. '
                    'Investigate before proceeding.')
else:
  print('\nVerified: all counts match Notebook 02 exactly. Safe to proceed.')

## 2. Load Finalised Data & Build the BUSI Split

Scans all three BUSI arms, verifies identical filenames and class labels across them, then builds one canonical image list used as the single source of truth for the split, reused identically by every arm downstream.

### 2.1 Data Verification


In [ ]:
BUSI_ARMS = ['BUSI_annotated', 'BUSI_inpainted', 'BUSI_masked']
BUSI_CLASSES = ['benign', 'malignant', 'normal']

def scan_busi_arm(arm):
  records = []
  for cls in BUSI_CLASSES:
    class_dir = LOCAL_DATA_DIR / 'BUSI' / arm / cls
    for img_path in sorted(class_dir.glob('*.png')):
      records.append({'image_name': img_path.name, 'class': cls})
  return pd.DataFrame(records)

arm_dfs = {arm: scan_busi_arm(arm) for arm in BUSI_ARMS}

for arm, df in arm_dfs.items():
  print(f'{arm}: {len(df)} images -- {df["class"].value_counts().to_dict()}')

In [ ]:
REFERENCE_ARM = 'BUSI_inpainted'

filename_sets = {arm: set(df['image_name']) for arm, df in arm_dfs.items()}
reference_set = filename_sets[REFERENCE_ARM]

set_mismatch = False
for arm, fset in filename_sets.items():
  if arm == REFERENCE_ARM:
    continue
  only_in_ref = reference_set - fset
  only_in_arm = fset - reference_set
  if only_in_ref or only_in_arm:
    set_mismatch = True
    print(f'FILENAME MISMATCH -- {REFERENCE_ARM} vs {arm}')
    if only_in_ref:
      print(f'  Only in {REFERENCE_ARM}: {sorted(only_in_ref)}')
    if only_in_arm:
      print(f'  Only in {arm}: {sorted(only_in_arm)}')

class_maps = {arm: dict(zip(df['image_name'], df['class'])) for arm, df in arm_dfs.items()}
ref_map = class_maps[REFERENCE_ARM]

label_mismatches = [
    (fname, REFERENCE_ARM, ref_map[fname], arm, cmap[fname])
    for arm, cmap in class_maps.items() if arm != REFERENCE_ARM
    for fname in cmap
    if fname in ref_map and ref_map[fname] != cmap[fname]
]

if label_mismatches:
  print(f'LABEL MISMATCH(ES): {len(label_mismatches)} found')
  for m in label_mismatches:
    print(f'  {m}')

if set_mismatch or label_mismatches:
  raise ValueError('BUSI arms are not perfectly aligned. Investigate before '
                    'proceeding with the splitting process.')

print('Verified: identical filenames and consistent class labels across all three arms.')

In [ ]:
busi_canonical = arm_dfs[REFERENCE_ARM].sort_values('image_name').reset_index(drop=True)

print(f'Canonical BUSI image list: {len(busi_canonical)} images')
print(busi_canonical['class'].value_counts())

### 2.2 Stratified 15% Test Split

Splits the canonical BUSI list into a frozen test set (15%) and a remaining pool (85%) for cross-validation. Computed once, saved to disk, never regenerated: every later step reads from these files rather than re-splitting.

In [ ]:
# Path definition
ROOT = Path('/content/drive/MyDrive/University/Masters/MSc_Thesis')
OUTPUTS = ROOT / 'outputs'
RESULTS = OUTPUTS / 'results'

print(f'RESULTS exists: {RESULTS.exists()}')

In [ ]:
test_path = RESULTS / 'busi_test_15.csv'
cv_pool_path = RESULTS / 'busi_cv_85_pool.csv'

if test_path.exists() and cv_pool_path.exists():
  busi_test = pd.read_csv(test_path)
  busi_cv_pool = pd.read_csv(cv_pool_path)
  print('Loaded existing test/CV-pool split from Drive -- not recomputed.')

else:
  busi_cv_pool, busi_test = train_test_split(
      busi_canonical, test_size=0.15, stratify=busi_canonical['class'], random_state=SEED,
  )
  busi_cv_pool = busi_cv_pool.sort_values('image_name').reset_index(drop=True)
  busi_test = busi_test.sort_values('image_name').reset_index(drop=True)

  overlap = set(busi_test['image_name']) & set(busi_cv_pool['image_name'])
  if overlap:
    raise ValueError(f'Test/CV overlap detected: {overlap}. Do not save -- investigate.')
  if len(busi_test) + len(busi_cv_pool) != len(busi_canonical):
    raise ValueError('Split does not account for every image -- investigate before saving.')

  busi_test.to_csv(test_path, index=False)
  busi_cv_pool.to_csv(cv_pool_path, index=False)
  print('Computed, verified, and saved new test/CV-pool split.')

print('\nTest set (frozen, 15%):')
print(busi_test['class'].value_counts())
print(f'Total: {len(busi_test)}')
print('\nCV pool (85%):')
print(busi_cv_pool['class'].value_counts())
print(f'Total: {len(busi_cv_pool)}')

### 2.3 Assign 5-Fold CV Splits

Adds a `fold` column (0-4) to the CV pool via stratified k-fold. Computed once, saved to disk. Every sweep trial and final CV run reads this column rather than recomputing folds independently.

In [ ]:
cv_85_path = RESULTS / 'busi_cv_85.csv'

if cv_85_path.exists():
  busi_cv_85 = pd.read_csv(cv_85_path)
  print('Loaded existing 5-fold assignment from Drive -- not recomputed.')

else:
  skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
  busi_cv_85 = busi_cv_pool.copy()
  busi_cv_85['fold'] = -1

  for fold_idx, (_, val_idx) in enumerate(skf.split(busi_cv_85, busi_cv_85['class'])):
    busi_cv_85.loc[busi_cv_85.index[val_idx], 'fold'] = fold_idx

  # Every row must have a real fold assignment; no leftover placeholders.
  if (busi_cv_85['fold'] == -1).any():
    raise ValueError('Some rows never got assigned a fold; investigate before proceeding.')

  # Each fold's held-out validation images must not appear in any other fold.
  fold_sets = {f: set(busi_cv_85.loc[busi_cv_85['fold'] == f, 'image_name']) for f in range(5)}
  for i in range(5):
    for j in range(i + 1, 5):
      overlap = fold_sets[i] & fold_sets[j]
      if overlap:
        raise ValueError(f'Fold {i} and fold {j} overlap: {overlap}')

  # Folds together must cover the full pool, no image dropped or duplicated.
  covered = set.union(*fold_sets.values())
  if covered != set(busi_cv_85['image_name']) or len(busi_cv_85) != sum(len(s) for s in fold_sets.values()):
    raise ValueError('Fold coverage does not match the CV pool exactly; investigate before proceeding.')

  busi_cv_85.to_csv(cv_85_path, index=False)
  print('Computed, verified, and saved new 5-fold assignment.')

print(busi_cv_85.groupby(['fold', 'class']).size().unstack())

## 3. Dataset & DataLoader Construction

### 3.1 Dataset Class

General-purpose Dataset: takes a dataframe of `(image_name, class)` pairs, a base directory, and a mode flag. Works for any of BUSI's three arms, or BUS-UCLM/BrEaST, by pointing `base_dir` at the right folder. Augmentation and normalisation are injected as transforms, not hardcoded, so eval-only usage (external datasets, the frozen test set) is just a different transform pipeline passed in, not a different class.

In [ ]:
CLASS_TO_IDX = {'benign': 0, 'malignant': 1, 'normal': 2}

class BreastUltrasoundDataset(Dataset):
  """
  General-purpose dataset for any single BUSI arm, or BUS-UCLM/BrEaST.

  df        : dataframe with 'image_name' and 'class' columns (a full dataset's
              listing, or a fold-filtered slice of one)
  base_dir  : directory containing per-class subfolders
  transform : torchvision transform pipeline; differs between train mode (augmentation
              + normalisation) and eval mode (normalisation only)
  """
  def __init__(self, df, base_dir, transform):
    self.df = df.reset_index(drop=True)
    self.base_dir = Path(base_dir)
    self.transform = transform

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    row = self.df.iloc[idx]
    img_path = self.base_dir / row['class'] /row['image_name']

    image = Image.open(img_path).convert('RGB')
    image = self.transform(image)

    label = CLASS_TO_IDX[row['class']]
    return image, label

### 3.2 Augmentation Pipeline

Five independent transforms (flip, rotation, brightness, contrast, zoom),
each gated by its own `p=0.55` draw so an image can receive some, all, or none per epoch. **Training data only**.

In [ ]:
AUGMENTATION_P = 0.55

def build_train_augmentations():
  """
  Five independently-gated augmentations, each firing at AUGMENTATION_P, matching
  the clinical-mimicry reasoning settled on for this experiment: flip (lesion
  laterality), rotation (probe-holding variation), brightness/contrast (gain/dynamic range
  variation across machines), zoom (probe depth/pressure variation). Magnitudes
  deliberately conservative: ultrasound is already high-contrast and speckle-noisy;
  aggressive jitter risks destroying the tissue signal itself, not just varying
  its appearance.
  """
  return transforms.Compose([
      transforms.RandomApply([transforms.RandomHorizontalFlip(p=1.0)], p=AUGMENTATION_P),
      transforms.RandomApply([transforms.RandomRotation(degrees=12)], p=AUGMENTATION_P),
      transforms.RandomApply([transforms.ColorJitter(brightness=0.05)], p=AUGMENTATION_P),
      transforms.RandomApply([transforms.ColorJitter(contrast=0.08)], p=AUGMENTATION_P),
      transforms.RandomApply([transforms.RandomAffine(degrees=0, scale=(0.93, 1.07))], p=AUGMENTATION_P),
  ])

In [ ]:
FIGURES_DIR = OUTPUTS / 'figures_training'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Build an augmented dataset on the training data
aug_transform = transforms.Compose([
    build_train_augmentations(),
    transforms.ToTensor(),
])

aug_ds = BreastUltrasoundDataset(
    df=busi_cv_85,
    base_dir=LOCAL_DATA_DIR / 'BUSI' / 'BUSI_inpainted',
    transform=aug_transform,
)

# Check the combinations of the transformations applied
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for i, ax in enumerate(axes):
  img, _ = aug_ds[0]
  ax.imshow(img.permute(1, 2, 0))
  ax.axis('off')
plt.savefig(FIGURES_DIR / 'augmentation_examples.png', dpi=300, bbox_inches='tight')
plt.suptitle(f'Same image, 6 draws -- {busi_cv_85.iloc[0]["image_name"]}')
plt.show()

### 3.3 Normalisation

TL uses fixed ImageNet statistics. Scratch uses statistics computed from the training data itself: per fold (avoids validation-set leakage) and per arm (pixel content genuinely differs between annotated/inpainted/masked).

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [ ]:
def compute_dataset_stats(df, base_dir, batch_size=32):
  """
  Computes per-channel mean/std over a set of images, for scratch-mode normalisation.
  Called fresh for each (arm, fold) training subset; not stored as a single global
  constant. Stats must reflect the real, unaugmented data distribution.
  """
  stats_transform = transforms.Compose([transforms.ToTensor()])
  stats_ds = BreastUltrasoundDataset(df=df, base_dir=base_dir, transform=stats_transform)
  stats_loader = DataLoader(stats_ds, batch_size=batch_size, shuffle=False, num_workers=2)

  channel_sum = torch.zeros(3)
  channel_sum_sq = torch.zeros(3)
  n_pixels = 0

  for images, _ in stats_loader:
    b, c, h, w = images.shape
    channel_sum += images.sum(dim=[0, 2, 3])
    channel_sum_sq += (images ** 2).sum(dim=[0, 2, 3])
    n_pixels += b * h * w

  mean = (channel_sum / n_pixels).tolist()
  std = torch.sqrt(channel_sum_sq / n_pixels - torch.tensor(mean) ** 2).tolist()

  return mean, std

**Note**: `batch_size` parameter in the `compute_dataset_stats` function describes the chunks of images loaded for the dataset stats computation. Not related to the `batch_size` parameter in training.

In [ ]:
def build_transform(mode, mean, std):
  """
  mode: 'train' (augmentation + normalisation) or 'eval' (normalise only).
  mean/std: IMAGENET_MEAN/STD, or compute_dataset_stats() output for scratch; caller
            decides which, based on training mode and, for scratch, the specific
            arm/fold subset being trained on.
  """
  steps = []
  if mode == 'train':
    steps.append(build_train_augmentations())
  steps.append(transforms.ToTensor())
  steps.append(transforms.Normalize(mean=mean, std=std))
  return transforms.Compose(steps)

### 3.4 Class Weights

Inverse-frequency class weights for the loss function, computed per fold's training subset (not once globally, considering fold-to-fold class counts vary slightly, per Section 2.3's table).

In [ ]:
MALIGNANT_BOOST = 2.25

def compute_class_weights(df, malignant_boost=MALIGNANT_BOOST):
  """
  Standard inverse-frequency weights: weight_c = n_samples / (n_classes * count_c).
  malignant_boost applies an additional multiplier on top of that, encoding extra
  clinical cost for a missed malignant case beyond pure frequency correction.
  Computed per fold's training subset, not stored globally. They must reflect
  the real, unaugmented data distribution.
  """
  counts = df['class'].value_counts()
  n_samples = len(df)
  n_classes = len(CLASS_TO_IDX)

  weights = torch.zeros(n_classes)
  for cls, idx in CLASS_TO_IDX.items():
    weights[idx] = n_samples / (n_classes * counts.get(cls, 0))

  weights[CLASS_TO_IDX['malignant']] *= malignant_boost
  return weights

### 3.5 DataLoader Construction

Wraps the `Dataset` in a `DataLoader`: batch size 16, `pin_memory` for faster GPU transfer, and a seeded `worker_init_fn` so multi-process data loading doesn't silently break reproducibility.

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 4

def seed_worker(worker_id):
  """
  Without it, each DataLoader worker process inherits an unseeded RNG state:
  augmentation draws would differ run-to-run even with the global seed set, since
  that seed never propagates into worker subprocesses on its own.
  """
  worker_seed = torch.initial_seed() % 2**32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

def build_dataloader(dataset, train):
  return DataLoader(
      dataset,
      batch_size=BATCH_SIZE,
      shuffle=train,
      num_workers=NUM_WORKERS,
      pin_memory=True,
      worker_init_fn=seed_worker,
      generator=torch.Generator().manual_seed(SEED),
  )

## 4. Architecture Setup

### 4.1 Architecture Loading & Structure Inspection

Loads all three architectures with pretrained weights and prints their top-level modules, to confirm real layer names before freezing/unfreezing step.
Structure is identical for TL and scratch, so one load per architecture is enough for inspection.

In [ ]:
resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

print('ResNet50 top-level modules:')
for name, module in resnet50.named_children():
  print(f'  {name}: {type(module).__name__}')

In [ ]:
efficientnet_b0 = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

print('EfficientNet-B0 top-level modules:')
for name, module in efficientnet_b0.named_children():
  print(f'  {name}: {type(module).__name__}')

print('\nEfficientNet-B0 features:')
for i, block in enumerate(efficientnet_b0.features):
  print(f'  features[{i}]: {type(block).__name__}')

print('\nEfficientNet-B0 classifier submodules:')
for i, layer in enumerate(efficientnet_b0.classifier):
  print(f'  classifier[{i}]: {layer}')

In [ ]:
swin_t = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)

print('Swin-Tiny top-level modules:')
for name, module in swin_t.named_children():
  print(f'  {name}: {type(module).__name__}')

print('\nSwin-Tiny features:')
for i, block in enumerate(swin_t.features):
  print(f'  features[{i}]: {type(block).__name__}')

### 4.2 Classifier Head Replacement

Replaces each architecture's ImageNet 1000-class head with a 3-class head. Applied identically whether the backbone is pretrained or randomly initialised.

In [ ]:
NUM_CLASSES = 3

def build_resnet50(pretrained, dropout):
  weights = models.ResNet50_Weights.DEFAULT if pretrained else None
  model = models.resnet50(weights=weights)
  model.fc = nn.Sequential(
      nn.Dropout(p=dropout),
      nn.Linear(model.fc.in_features, NUM_CLASSES),
  )
  return model

def build_efficientnet_b0(pretrained, dropout):
  weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
  model = models.efficientnet_b0(weights=weights)
  model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
  model.classifier[0].p = dropout
  return model

def build_swin_t(pretrained, dropout):
  weights = models.Swin_T_Weights.DEFAULT if pretrained else None
  model = models.swin_t(weights=weights, dropout=dropout)
  model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
  return model

ARCHITECTURE_BUILDERS = {
    'resnet50': build_resnet50,
    'efficientnet_b0': build_efficientnet_b0,
    'swin_tiny': build_swin_t,
}

In [ ]:
def get_output_linear(model):
  """
  Returns the actual Linear output layer, regardless of whether it's
  wrapped in a Sequential (ResNet's fc, EfficientNet's classifier) or
  standalone (Swin's head).
  """
  candidate = model.fc if hasattr(model, 'fc') else (
      model.classifier if hasattr(model, 'classifier') else model.head
  )
  if isinstance(candidate, nn.Sequential):
    linear_layers = [m for m in candidate if isinstance(m, nn.Linear)]
    return linear_layers[-1]
  return candidate


for name, builder in ARCHITECTURE_BUILDERS.items():
  model = builder(pretrained=True, dropout=0.3)
  final_layer = get_output_linear(model)
  print(f'{name}: final layer out_features: {final_layer.out_features} (expected: {NUM_CLASSES})')

### 4.3 Freeze/Unfreeze Utility (Transfer Learning Only)

Freezes backbone parameters according to one of three categories: head only, last 1 block, last 2 blocks, per the sweep's unfrozen-layers dimension.

In [ ]:
def freeze_resnet50(model, category):
  for param in model.parameters():
    param.requires_grad = False

  for param in model.fc.parameters():
    param.requires_grad = True

  if category == 'last_1_block':
    for param in model.layer4.parameters():
      param.requires_grad = True
  elif category == 'last_2_blocks':
    for param in model.layer3.parameters():
      param.requires_grad = True
    for param in model.layer4.parameters():
      param.requires_grad = True
  elif category != 'head_only':
    raise ValueError(f'Unknown category: {category}')

  return model

In [ ]:
def freeze_efficientnet_b0(model, category):
  for param in model.parameters():
    param.requires_grad = False

  for param in model.classifier.parameters():
    param.requires_grad = True

  if category == 'last_1_block':
    for param in model.features[7].parameters():
      param.requires_grad = True
    for param in model.features[8].parameters():
      param.requires_grad = True
  elif category == 'last_2_blocks':
    for param in model.features[6].parameters():
      param.requires_grad = True
    for param in model.features[7].parameters():
      param.requires_grad = True
    for param in model.features[8].parameters():
      param.requires_grad = True
  elif category != 'head_only':
    raise ValueError(f'Unknown category: {category}')

  return model

In [ ]:
def freeze_swin_t(model, category):
  for param in model.parameters():
    param.requires_grad = False

  for param in model.head.parameters():
    param.requires_grad = True

  if category == 'last_1_block':
    for param in model.features[7].parameters():
      param.requires_grad = True
  elif category == 'last_2_blocks':
    for param in model.features[5].parameters():
      param.requires_grad = True
    for param in model.features[6].parameters():
      param.requires_grad = True
    for param in model.features[7].parameters():
      param.requires_grad = True
  elif category != 'head_only':
    raise ValueError(f'Unknown category: {category}')

  return model

In [ ]:
FREEZE_FUNCTIONS = {
    'resnet50': freeze_resnet50,
    'efficientnet_b0': freeze_efficientnet_b0,
    'swin_tiny': freeze_swin_t,
}

In [ ]:
for arch_name, builder in ARCHITECTURE_BUILDERS.items():
  freeze_fn = FREEZE_FUNCTIONS[arch_name]
  for category in ['head_only', 'last_1_block', 'last_2_blocks']:
    model = builder(pretrained=True, dropout=0.3)
    model = freeze_fn(model, category)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'{arch_name} / {category}: {trainable:,} / {total:,} trainable ({100*trainable/total:.1f}%)')

### 4.4 Unified Model Builder

Single entry point used by the sweep and CV loop: builds an architecture, applies the correct classifier head, and freezes per the unfrozen-layers category if in TL mode.

In [ ]:
def build_model(architecture, mode, dropout, unfreeze_category=None):
  """
  architecture        : 'resnet50' | 'efficientnet_b0' | 'swin_tiny'
  mode                : 'TL' | 'scratch'
  unfreeze_category   : 'head_only' | 'last_1_block' | 'last_2_blocks'
                          required if mode='TL', ignored (must be None)
                          if mode='scratch'.
  """
  if mode == 'TL':
    if unfreeze_category is None:
      raise ValueError("mode='TL' requires an unfreeze_category.")
    model = ARCHITECTURE_BUILDERS[architecture](pretrained=True, dropout=dropout)
    model = FREEZE_FUNCTIONS[architecture](model, unfreeze_category)
  elif mode == 'scratch':
    if unfreeze_category is not None:
      raise ValueError("mode='scratch' does not use unfreeze_category.")
    model = ARCHITECTURE_BUILDERS[architecture](pretrained=False, dropout=dropout)
  else:
    raise ValueError(f'Unknown mode: {mode}')

  return model

## 5. Training Engine

### 5.1 Optimiser & Scheduler

AdamW for TL and ViT-scratch; SGD+momentum+cosine for CNN-scratch. ViT modes get a shorter linear warmup before cosine decay.

In [ ]:
def build_optimizer(model, architecture, mode, lr, weight_decay):
  if mode == 'scratch' and architecture != 'swin_tiny':
    return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
  return optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

def build_scheduler(optimizer, architecture, mode, epochs, warmup_epochs=5):
  if architecture == 'swin_tiny':
    warmup = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_epochs)
    cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    return optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])
  else:
    return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [ ]:
EPOCH_CEILINGS = {
    ('resnet50', 'TL'): 50, ('resnet50', 'scratch'): 100,
    ('efficientnet_b0', 'TL'): 50, ('efficientnet_b0', 'scratch'): 100,
    ('swin_tiny', 'TL'): 60, ('swin_tiny', 'scratch'): 200,
}

for (arch, mode), epochs in EPOCH_CEILINGS.items():
  model = build_model(arch, mode, dropout=0.3, unfreeze_category='head_only' if mode == 'TL' else None)
  opt = build_optimizer(model, arch, mode, lr=1e-4, weight_decay=1e-4)
  sched = build_scheduler(opt, arch, mode, epochs)
  print(f'{arch} / {mode}: optimizer={type(opt).__name__}, scheduler={type(sched).__name__ if sched else "None (constant LR)"}')

### 5.2 Training-Time Metric: Macro-F2

Lightweight macro-averaged F2, computed once per epoch from that epoch's validation predictions. Drives early stopping and the sweep objective. The full evaluation metric is separate and more detailed.

In [ ]:
def macro_f2(y_true, y_pred, num_classes=NUM_CLASSES, beta=2):
  """
  Macro-averaged F-beta (beta=2 -> F2, recall weighted above precision).
  Computed per class, then averaged unweighted across classes, so that the smallest
  class counts equally to the largest, matching the imbalance-aware framing used
  throughout this notebook's evaluation design.
  """
  y_true = np.asarray(y_true)
  y_pred = np.asarray(y_pred)

  f2_per_class = []
  for cls in range(num_classes):
    tp = np.sum((y_pred == cls) & (y_true == cls))
    fp = np.sum((y_pred == cls) & (y_true != cls))
    fn = np.sum((y_pred != cls) & (y_true == cls))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if precision + recall == 0:
      f2 = 0.0
    else:
      f2 = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall)

    f2_per_class.append(f2)

  return np.mean(f2_per_class)

In [ ]:
# Constructed case with a known correct answer (worked out by hand):
# macro-F2 = 0.6111. Verifies the function before it drives early stopping
# and the sweep objective on real data.
y_true_test = np.array([0, 0, 1, 1, 2, 2])
y_pred_test = np.array([0, 0, 0, 0, 2, 2])

score = macro_f2(y_true_test, y_pred_test)
print(f'Constructed test case macro-F2: {score:.4f}')

### 5.3 Training & Validation Loop

Core forward/backward step and no-grad evaluation step, both AMP-wrapped. Scheduler stepping happens once per epoch at the outer loop level.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device):
  model.train()
  running_loss = 0.0
  n_samples = 0

  for images, labels in dataloader:
    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    optimizer.zero_grad()

    with autocast('cuda'):
      outputs = model(images)
      loss = criterion(outputs, labels)

    # scaler.step skips the optimizer update entirely if gradients overflow
    # to inf/NaN under fp16.
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    running_loss += loss.item() * images.size(0)
    n_samples += images.size(0)

  return running_loss / n_samples

In [ ]:
def validate_one_epoch(model, dataloader, criterion, device):
  model.eval()
  running_loss = 0.0
  n_samples = 0
  all_preds, all_labels = [], []

  with torch.no_grad():
    for images, labels in dataloader:
      images = images.to(device, non_blocking=True)
      labels = labels.to(device, non_blocking=True)

      with autocast('cuda'):
        outputs = model(images)
        loss = criterion(outputs, labels)

      running_loss += loss.item() * images.size(0)
      n_samples += images.size(0)
      all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  val_loss = running_loss / n_samples
  val_f2 = macro_f2(all_labels, all_preds)
  return val_loss, val_f2

### 5.4 Early Stopping

Tracks a 3-epoch moving average of validation macro-F2. Stops training if that smoothed metric hasn't improved for `patience` epochs, but never before `grace_period` epochs have passed. The epoch ceiling itself is enforced by the outer training loop.

In [ ]:
PATIENCE_BY_MODE = {
    ('resnet50', 'TL'): 8,        ('resnet50', 'scratch'): 12,
    ('efficientnet_b0', 'TL'): 8, ('efficientnet_b0', 'scratch'): 12,
    ('swin_tiny', 'TL'): 8,       ('swin_tiny', 'scratch'): 25,
}

GRACE_PERIOD_BY_MODE = {
    ('resnet50', 'TL'): 10,        ('resnet50', 'scratch'): 15,
    ('efficientnet_b0', 'TL'): 10, ('efficientnet_b0', 'scratch'): 15,
    ('swin_tiny', 'TL'): 10,       ('swin_tiny', 'scratch'): 20,
}

class EarlyStopper:
  """
  Stops training when the 3-epoch moving average of validation macro-F2 hasn't
  improved for `patience` epochs, but never triggers before `grace_period` epochs
  have passed, regardless of patience. Smoothing absorbs single-epoch noise from
  small validation folds; the grace period protects against volatile early epochs
  being mistaken for a genuine plateau.
  """
  def __init__(self, patience, grace_period, window=3):
    self.patience = patience
    self.grace_period = grace_period
    self.window = window
    self.history = []
    self.best_smoothed = -float('inf')
    self.epochs_without_improvement = 0
    self.stopped_epoch = None

  def step(self, val_f2, current_epoch):
    self.history.append(val_f2)
    smoothed = np.mean(self.history[-self.window:])

    if current_epoch < self.grace_period:
      if smoothed > self.best_smoothed:
        self.best_smoothed = smoothed
      return False

    if smoothed > self.best_smoothed:
      self.best_smoothed = smoothed
      self.epochs_without_improvement = 0
    else:
      self.epochs_without_improvement += 1

    if self.epochs_without_improvement >= self.patience:
      self.stopped_epoch = current_epoch
      return True

    return False

In [ ]:
# Scripted val-F2 sequence with a known-correct expected outcome:
# grace period absorbs the epoch-7 noise dip entirely; geniuine plateau from epoch
# 10 triggers a stop around epoch 18. Verifies the stopping logic.
scripted_f2 = [0.30, 0.35, 0.40, 0.44, 0.47, 0.49,
               0.50, 0.30, 0.50, 0.50,
               0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50, 0.50]

stopper = EarlyStopper(patience=8, grace_period=10)

for epoch, f2 in enumerate(scripted_f2):
  should_stop = stopper.step(f2, epoch)
  print(f'Epoch {epoch}: val_f2={f2:.2f}, smoothed_best={stopper.best_smoothed:.3f}, '
        f'no_improve={stopper.epochs_without_improvement}, stop={should_stop}')
  if should_stop:
    break

### 5.5 Checkpointing

Two checkpoint types: "latest" (full state, for crash resume, deleted on successful completion) and "best" (weights only, kept permanently). Sweep trials use "latest" alone; final runs use both.

In [ ]:
def save_latest_checkpoint(path, model, optimizer, scheduler, epoch, stopper, scaler, wandb_run_id=None):
  torch.save({
      'epoch': epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
      'scaler_state_dict': scaler.state_dict(),
      'stopper_history': stopper.history,
      'stopper_best_smoothed': stopper.best_smoothed,
      'stopper_epochs_without_improvement': stopper.epochs_without_improvement,
      'rng_state_torch_cpu': torch.get_rng_state(),
      'rng_state_torch_cuda': torch.cuda.get_rng_state(),
      'rng_state_numpy': np.random.get_state(),
      'rng_state_python': random.getstate(),
      'wandb_run_id': wandb_run_id,
  }, path)


def load_latest_checkpoint(path, model, optimizer, scheduler, scaler, stopper):
  ckpt = torch.load(path, map_location=device, weights_only=False)

  model.load_state_dict(ckpt['model_state_dict'])
  optimizer.load_state_dict(ckpt['optimizer_state_dict'])
  if scheduler and ckpt['scheduler_state_dict']:
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
  scaler.load_state_dict(ckpt['scaler_state_dict'])

  stopper.history = ckpt['stopper_history']
  stopper.best_smoothed = ckpt['stopper_best_smoothed']
  stopper.epochs_without_improvement = ckpt['stopper_epochs_without_improvement']

  # Both explicitly forced to CPU: map_location moves every tensor in the
  # checkpoint to `device` during load, but PyTorch's RNG state setters
  # require CPU ByteTensors regardless of which device training runs on.
  torch.set_rng_state(ckpt['rng_state_torch_cpu'].cpu())
  torch.cuda.set_rng_state(ckpt['rng_state_torch_cuda'].cpu())
  np.random.set_state(ckpt['rng_state_numpy'])
  random.setstate(ckpt['rng_state_python'])

  return ckpt['epoch'] + 1, ckpt.get('wandb_run_id')


def save_best_checkpoint(path, model):
  """Weights only (for evaluation and GradCAM)."""
  torch.save(model.state_dict(), path)

In [ ]:
# Round-trip test: save real state, deliberately corrupt the live model,
# reload, confirm the corruption is undone. Proves the checkpoint mechanism
# actually restores correct state.
test_model = build_model('resnet50', mode='TL', dropout=0.3, unfreeze_category='head_only').to(device)
test_optimizer = build_optimizer(test_model, 'resnet50', 'TL', lr=1e-4, weight_decay=1e-4)
test_scheduler = build_scheduler(test_optimizer, 'resnet50', 'TL', epochs=50)
test_scaler = GradScaler('cuda')
test_stopper = EarlyStopper(patience=8, grace_period=10)

for f2 in [0.30, 0.40, 0.45]:
  test_stopper.step(f2, len(test_stopper.history))

test_path = Path('/content/test_checkpoint.pt')
save_latest_checkpoint(test_path, test_model, test_optimizer, test_scheduler, epoch=2,
                        stopper=test_stopper, scaler=test_scaler)

original_weight = get_output_linear(test_model).weight.data.clone()

with torch.no_grad():
  get_output_linear(test_model).weight.data += 999.0

fresh_stopper = EarlyStopper(patience=8, grace_period=10)
resumed_epoch, _ = load_latest_checkpoint(test_path, test_model, test_optimizer,
                                           test_scheduler, test_scaler, fresh_stopper)

weights_match = torch.allclose(get_output_linear(test_model).weight.data, original_weight)
history_match = fresh_stopper.history == test_stopper.history

print(f'Resumed epoch: {resumed_epoch} (expected: 3)')
print(f'Weights correctly restored after corruption: {weights_match}')
print(f'Stopper history correctly restored: {history_match} -- {fresh_stopper.history}')

test_path.unlink()

In [ ]:
best_path = Path('/content/test_best.pt')
save_best_checkpoint(best_path, test_model)

fresh_model = build_model('resnet50', mode='TL', dropout=0.3, unfreeze_category='head_only').to(device)
fresh_model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))

matches = torch.allclose(get_output_linear(fresh_model).weight.data, get_output_linear(test_model).weight.data)
print(f'Best checkpoint round-trip correct: {matches}')

best_path.unlink()

### 5.6 Unified Training Entry Point

Ties 5.1-5.5 together: builds optimiser/scheduler, runs the epoch loop with early stopping and checkpointing, logs per-epoch metrics to WandB. Single call site for both sweep trials and final CV/retrain runs.

In [ ]:
def train_model(architecture, mode, dropout, train_loader, val_loader,
                 class_weights, lr, weight_decay, epochs, patience, grace_period,
                 checkpoint_dir, run_identity, wandb_config, *,
                 unfreeze_category=None, resume=False):
  """
  Full training run: builds optimizer/scheduler fresh, loops epochs with
  early stopping, checkpoints per 5.5's hybrid scheme, logs to WandB.

  checkpoint_dir : where 'latest.pt' and 'best.pt' are written for THIS run
  resume         : if True and a 'latest.pt' already exists in
                    checkpoint_dir, resumes from it instead of starting
                    fresh.
  wandb_config    : full hyperparameter dict for this run, logged once at
                    wandb.init().
  """
  seed_everything(SEED)

  checkpoint_dir = Path(checkpoint_dir)
  checkpoint_dir.mkdir(parents=True, exist_ok=True)
  latest_path = checkpoint_dir / 'latest.pt'
  best_path = checkpoint_dir / 'best.pt'

  model = build_model(architecture, mode, dropout, unfreeze_category=unfreeze_category).to(device)
  optimizer = build_optimizer(model, architecture, mode, lr, weight_decay)
  scheduler = build_scheduler(optimizer, architecture, mode, epochs)
  scaler = GradScaler('cuda')
  stopper = EarlyStopper(patience=patience, grace_period=grace_period)
  criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

  start_epoch = 0
  wandb_run_id = None
  if resume and latest_path.exists():
    start_epoch, wandb_run_id = load_latest_checkpoint(latest_path, model, optimizer, scheduler, scaler, stopper)
    print(f'Resumed from checkpoint at epoch {start_epoch}')

  if wandb_run_id:
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                      id=wandb_run_id, resume='must', settings=wandb.Settings(silent=True),
                      **run_identity)
  else:
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                      config=wandb_config, resume='allow', settings=wandb.Settings(silent=True),
                      **run_identity)

  best_f2_epoch = start_epoch
  for epoch in range(start_epoch, epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_f2 = validate_one_epoch(model, val_loader, criterion, device)

    if scheduler:
      scheduler.step()

    should_stop = stopper.step(val_f2, epoch)

    if val_f2 >= stopper.best_smoothed:
      save_best_checkpoint(best_path, model)
      best_f2_epoch = epoch

    save_latest_checkpoint(latest_path, model, optimizer, scheduler, epoch, stopper, scaler,
                            wandb_run_id=run.id)

    wandb.log({
        'epoch': epoch, 'train_loss': train_loss,
        'val_loss': val_loss, 'val_f2': val_f2,
        'val_f2_smoothed': np.mean(stopper.history[-3:]),
        'lr': optimizer.param_groups[0]['lr'],
    })

    if should_stop:
      print(f'Early stopping triggered at epoch {epoch}')
      break

  latest_path.unlink()
  run.finish()

  return {
      'final_epoch': epoch,
      'best_f2_epoch': best_f2_epoch,
      'best_smoothed_f2': stopper.best_smoothed,
      'best_checkpoint_path': str(best_path),
      'wandb_run_id': run.id,
  }

#### Integration test

Check whether the checkpoints get triggered correctly on full and interrupted runs.

In [ ]:
integ_train_df = busi_cv_85[busi_cv_85['fold'] != 0].groupby('class').sample(n=10, random_state=SEED)
integ_val_df = busi_cv_85[busi_cv_85['fold'] == 0].groupby('class').sample(n=4, random_state=SEED)

integ_train_ds = BreastUltrasoundDataset(integ_train_df, LOCAL_DATA_DIR / 'BUSI' / 'BUSI_inpainted',
                                          build_transform('train', IMAGENET_MEAN, IMAGENET_STD))
integ_val_ds = BreastUltrasoundDataset(integ_val_df, LOCAL_DATA_DIR / 'BUSI' / 'BUSI_inpainted',
                                        build_transform('eval', IMAGENET_MEAN, IMAGENET_STD))

integ_train_loader = build_dataloader(integ_train_ds, train=True)
integ_val_loader = build_dataloader(integ_val_ds, train=False)
integ_class_weights = compute_class_weights(integ_train_df, malignant_boost=MALIGNANT_BOOST)

In [ ]:
# Integration test, Part A: normal uninterrupted completion. Confirms
# train_model's full lifecycle: optimiser/scheduler build, epoch loop, early stopping,
# checkpoint save/cleanup, WandB logging on real small data sample.
partA_identity = build_run_identity('resnet50', 'TL', 'inpainted', job_type='sweep')
partA_identity['tags'].append('integration_test')
partA_identity['name'] += '_integration_test_A'

partA_dir = Path('/content/test_train_model_A')

partA_result = train_model(
    'resnet50', 'TL', 0.3, integ_train_loader, integ_val_loader,
    integ_class_weights, lr=1e-4, weight_decay=1e-4, epochs=3,
    patience=8, grace_period=10, checkpoint_dir=partA_dir,
    run_identity=partA_identity,
    wandb_config={'lr': 1e-4, 'weight_decay': 1e-4, 'integration_test': True},
    unfreeze_category='head_only',
)

print(f'Part A result: {partA_result}')
print(f'latest.pt deleted after completion: {not (partA_dir / "latest.pt").exists()}')
print(f'best.pt exists: {(partA_dir / "best.pt").exists()}')

In [ ]:
# Integration test, Part B: simulated crash mid-training (train_one_epoch
# monkey-patched to fail on its 2nd call), then resume. Confirms local state
# resumes from the correct epoch AND the resumed run reattaches to the same
# WandB run rather than fragmenting into a second, disconnected one.
original_train_one_epoch = train_one_epoch

def crashing_train_one_epoch(*args, **kwargs):
  crashing_train_one_epoch.calls += 1
  if crashing_train_one_epoch.calls == 2:
    raise RuntimeError('Simulated crash for integration test')
  return original_train_one_epoch(*args, **kwargs)
crashing_train_one_epoch.calls = 0

partB_identity = build_run_identity('resnet50', 'TL', 'inpainted', job_type='sweep')
partB_identity['tags'].append('integration_test')
partB_identity['name'] += '_integration_test_B'

partB_dir = Path('/content/test_train_model_B')

train_one_epoch = crashing_train_one_epoch
try:
  train_model(
      'resnet50', 'TL', 0.3, integ_train_loader, integ_val_loader,
      integ_class_weights, lr=1e-4, weight_decay=1e-4, epochs=5,
      patience=8, grace_period=10, checkpoint_dir=partB_dir,
      run_identity=partB_identity,
      wandb_config={'lr': 1e-4, 'weight_decay': 1e-4, 'integration_test': True},
      unfreeze_category='head_only',
  )
except RuntimeError as e:
  print(f'Simulated crash caught as expected: {e}')
finally:
  train_one_epoch = original_train_one_epoch

print(f'latest.pt survived the crash: {(partB_dir / "latest.pt").exists()}')

crashed_ckpt = torch.load(partB_dir / 'latest.pt', map_location=device, weights_only=False)
crashed_run_id = crashed_ckpt['wandb_run_id']
print(f'WandB run id at crash: {crashed_run_id}')

partB_resumed_result = train_model(
    'resnet50', 'TL', 0.3, integ_train_loader, integ_val_loader,
    integ_class_weights, lr=1e-4, weight_decay=1e-4, epochs=5,
    patience=8, grace_period=10, checkpoint_dir=partB_dir,
    run_identity=partB_identity,
    wandb_config={'lr': 1e-4, 'weight_decay': 1e-4, 'integration_test': True},
    unfreeze_category='head_only', resume=True,
)

print(f'Resumed result: {partB_resumed_result}')
print(f'Final epoch reached ceiling (expected 4): {partB_resumed_result["final_epoch"]}')
print(f'latest.pt deleted after completion: {not (partB_dir / "latest.pt").exists()}')
print(f'Resumed run reattached to SAME wandb run: {partB_resumed_result["wandb_run_id"] == crashed_run_id}')

In [ ]:
# Remove the saved checkpoints from Colab
shutil.rmtree(partA_dir, ignore_errors=True)
shutil.rmtree(partB_dir, ignore_errors=True)

## 6. Hyperparameter Sweep

### 6.1 Shared K-Fold CV Runner

`run_kfold_cv()` runs 5-fold CV for one architecture/mode/arm/hyperparameter combination. `context=final` means that each fold is its own checkpointed WandB run via `train_model()`. `context=sweep` means folds log into one already-open trial run, no checkpointing; sweep trials are disposable.

In [ ]:
CHECKPOINTS = ROOT / 'checkpoints'

CHECKPOINT_ARCH_NAMES = {
    'resnet50': 'resnet50',
    'efficientnet_b0': 'efficientnet',
    'swin_tiny': 'swin_transformer',
}
CHECKPOINT_MODE_NAMES = {
    'TL': 'transfer_learning',
    'scratch': 'from_scratch',
}

In [ ]:
def _train_fold_no_checkpoint(architecture, mode, dropout, train_loader, val_loader,
                               class_weights, lr, weight_decay, epochs, patience,
                               grace_period, *, unfreeze_category=None, log_fn=None):
  """
  Same core mechanics as train_model() -- no WandB run management, no
  checkpointing. Used only for sweep trials, which need no resume per the
  hybrid checkpointing design.
  """
  model = build_model(architecture, mode, dropout, unfreeze_category=unfreeze_category).to(device)
  optimizer = build_optimizer(model, architecture, mode, lr, weight_decay)
  scheduler = build_scheduler(optimizer, architecture, mode, epochs)
  scaler = GradScaler('cuda')
  stopper = EarlyStopper(patience=patience, grace_period=grace_period)
  criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

  best_f2_epoch = 0
  for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_f2 = validate_one_epoch(model, val_loader, criterion, device)
    if scheduler:
      scheduler.step()

    should_stop = stopper.step(val_f2, epoch)
    if val_f2 >= stopper.best_smoothed:
      best_f2_epoch = epoch

    if log_fn:
      log_fn({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
              'val_f2': val_f2, 'val_f2_smoothed': float(np.mean(stopper.history[-3:]))})

    if should_stop:
      break

  return {'final_epoch': epoch, 'best_f2_epoch': best_f2_epoch, 'best_smoothed_f2': stopper.best_smoothed}

In [ ]:
def run_kfold_cv(architecture, mode, arm, dropout, lr, weight_decay,
                  epochs, patience, grace_period, *,
                  unfreeze_category=None, context='final', trial_run=None):
  """
  context='final' : each fold -> train_model(), own checkpointed WandB run.
  context='sweep' : each fold -> _train_fold_no_checkpoint(), logs into the
                     already-open trial_run via fold-prefixed keys.
  Returns per-fold results (val_f2, stopping epoch) and their average,
  identically shaped regardless of context.
  """
  fold_results = []
  base_dir = LOCAL_DATA_DIR / 'BUSI' / f'BUSI_{arm}'

  for fold in range(5):
    train_df = busi_cv_85[busi_cv_85['fold'] != fold]
    val_df = busi_cv_85[busi_cv_85['fold'] == fold]

    if mode == 'TL':
      mean, std = IMAGENET_MEAN, IMAGENET_STD
    else:
      mean, std = compute_dataset_stats(train_df, base_dir)

    train_ds = BreastUltrasoundDataset(train_df, base_dir, build_transform('train', mean, std))
    val_ds = BreastUltrasoundDataset(val_df, base_dir, build_transform('eval', mean, std))
    train_loader = build_dataloader(train_ds, train=True)
    val_loader = build_dataloader(val_ds, train=False)
    class_weights = compute_class_weights(train_df, malignant_boost=MALIGNANT_BOOST)

    if context == 'final':
      identity = build_run_identity(architecture, mode, arm, job_type='cv_fold', fold=fold)
      wandb_config = {
          'architecture': architecture, 'mode': mode, 'arm': arm, 'fold': fold,
          'dropout': dropout, 'lr': lr, 'weight_decay': weight_decay,
          'unfreeze_category': unfreeze_category,
      }
      checkpoint_dir = (CHECKPOINTS / CHECKPOINT_ARCH_NAMES[architecture] /
                         CHECKPOINT_MODE_NAMES[mode] / f'{arm}_fold{fold}')
      result = train_model(
          architecture, mode, dropout, train_loader, val_loader, class_weights,
          lr, weight_decay, epochs, patience, grace_period,
          checkpoint_dir=checkpoint_dir, run_identity=identity, wandb_config=wandb_config,
          unfreeze_category=unfreeze_category, resume=True,
      )
      fold_results.append({'fold': fold, 'val_f2': result['best_smoothed_f2'],
                            'stopping_epoch': result['final_epoch']})

    elif context == 'sweep':
      if trial_run is None:
        raise ValueError("context='sweep' requires an active trial_run.")
      seed_everything(SEED)
      result = _train_fold_no_checkpoint(
          architecture, mode, dropout, train_loader, val_loader, class_weights,
          lr, weight_decay, epochs, patience, grace_period,
          unfreeze_category=unfreeze_category,
          log_fn=lambda m, f=fold: trial_run.log({f'fold_{f}/{k}': v for k, v in m.items()}),
      )
      fold_results.append({'fold': fold, 'val_f2': result['best_smoothed_f2'],
                            'stopping_epoch': result['final_epoch']})

    else:
      raise ValueError(f'Unknown context: {context}')

  avg_val_f2 = float(np.mean([r['val_f2'] for r in fold_results]))
  return {'fold_results': fold_results, 'avg_val_f2': avg_val_f2}

### 6.2 Sweep Search-Space Configurations

Six WandB sweep configs, one per architecture × mode. LR range mirrors `build_optimizer`'s SGD/AdamW split. `unfreeze_category` only appears for TL. Fixed, non-swept values (epochs, patience, grace period, architecture, mode) are included as WandB `value` parameters, so every trial's logged config is self-contained and readable from the dashboard alone.

In [ ]:
NUM_TRIALS_PER_SWEEP = 30

def get_lr_range(architecture, mode):
  """
  Mirrors build_optimizer's condition exactly: CNN-scratch uses SGD at a
  higher range (linear-scaling estimate from ResNet's original recipe,
  locked earlier); every other combination uses AdamW at the lower,
  fine-tuning-appropriate range.
  """
  if mode == 'scratch' and architecture != 'swin_tiny':
    return 1e-3, 1e-2
  return 1e-5, 1e-3


def build_sweep_config(architecture, mode):
  lr_min, lr_max = get_lr_range(architecture, mode)

  parameters = {
      'lr': {'distribution': 'log_uniform_values', 'min': lr_min, 'max': lr_max},
      'weight_decay': {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 1e-2},
      'dropout': {'distribution': 'uniform', 'min': 0.1, 'max': 0.5},
      'architecture': {'value': architecture},
      'mode': {'value': mode},
      'epochs': {'value': EPOCH_CEILINGS[(architecture, mode)]},
      'patience': {'value': PATIENCE_BY_MODE[(architecture, mode)]},
      'grace_period': {'value': GRACE_PERIOD_BY_MODE[(architecture, mode)]},
  }
  if mode == 'TL':
    parameters['unfreeze_category'] = {'values': ['head_only', 'last_1_block', 'last_2_blocks']}

  return {
      'name': f'{architecture}_{mode}_sweep',
      'method': 'bayes',
      'metric': {'name': 'avg_val_f2', 'goal': 'maximize'},
      'parameters': parameters,
  }


SWEEP_COMBINATIONS = [
    ('resnet50', 'TL'), ('resnet50', 'scratch'),
    ('efficientnet_b0', 'TL'), ('efficientnet_b0', 'scratch'),
    ('swin_tiny', 'TL'), ('swin_tiny', 'scratch'),
]

SWEEP_CONFIGS = {combo: build_sweep_config(*combo) for combo in SWEEP_COMBINATIONS}

In [ ]:
for (arch, mode), cfg in SWEEP_CONFIGS.items():
  print(f'{arch} / {mode}:')
  for pname, pspec in cfg['parameters'].items():
    print(f'  {pname}: {pspec}')
  print()

### 6.3 Sweep Trial Function

One trial function, bound per architecture/mode via closure so run identity (group, job_type, tags) is set correctly at `wandb.init()` time. Reads sampled hyperparameters from `wandb.config`, runs 5-fold CV via `run_kfold_cv (context=sweep)`, logs per-fold and averaged results. Exceptions are caught and logged, never re-raised, so one failed trial cannot halt the sweep.

In [ ]:
def make_sweep_trial_fn(architecture, mode, arm='inpainted'):
  """
  Returns a zero-argument function bound to one architecture/mode, suitable
  for wandb.agent(). Binding architecture/mode via closure, rather than
  reading them only from wandb.config, lets group/job_type/tags be set
  correctly in the same wandb.init() call, with no reliance on mutating
  them afterward.
  """
  identity_base = build_run_identity(architecture, mode, arm, job_type='sweep')

  def sweep_trial():
    run = wandb.init(
        group=identity_base['group'], job_type=identity_base['job_type'],
        tags=identity_base['tags'], settings=wandb.Settings(silent=True),
    )
    run.name = f"{identity_base['name']}_{run.id}"

    config = run.config
    # Defensive check: catches a mismatched sweep_id/agent pairing
    # immediately, rather than silently training the wrong architecture.
    assert config.architecture == architecture and config.mode == mode, (
        f'Sweep config ({config.architecture}/{config.mode}) does not match '
        f'bound trial function ({architecture}/{mode}).'
    )

    trial_failed = False
    try:
      result = run_kfold_cv(
          architecture=architecture, mode=mode, arm=arm,
          dropout=config.dropout, lr=config.lr, weight_decay=config.weight_decay,
          epochs=config.epochs, patience=config.patience, grace_period=config.grace_period,
          unfreeze_category=config.get('unfreeze_category'),
          context='sweep', trial_run=run,
      )

      for r in result['fold_results']:
        run.log({f"fold_{r['fold']}/final_stopping_epoch": r['stopping_epoch'],
                  f"fold_{r['fold']}/final_val_f2": r['val_f2']})
      run.log({'avg_val_f2': result['avg_val_f2']})

      print(f'Trial complete -- {architecture}/{mode}: avg_val_f2={result["avg_val_f2"]:.4f}')

    except Exception as e:
      trial_failed = True
      print(f'Trial FAILED -- {architecture}/{mode}: {e}')

    finally:
      run.finish(exit_code=1 if trial_failed else 0)

  return sweep_trial

### 6.4 Launching the Sweeps

Six independent cells, one per architecture × mode, run manually. Each is idempotent: re-running a cell reuses its existing sweep rather than creating a duplicate, and automatically launches only the trials still remaining out of 30, making it safe to re-run after any disconnect without manual bookkeeping.

In [ ]:
SWEEP_IDS_PATH = RESULTS / 'sweep_ids.json'

def load_sweep_ids():
  if SWEEP_IDS_PATH.exists():
    with open(SWEEP_IDS_PATH) as f:
      return json.load(f)
  return {}

def save_sweep_id(architecture, mode, sweep_id):
  ids = load_sweep_ids()
  ids[f'{architecture}_{mode}'] = sweep_id
  with open(SWEEP_IDS_PATH, 'w') as f:
    json.dump(ids, f, indent=2)

def get_or_create_sweep_id(architecture, mode):
  ids = load_sweep_ids()
  key = f'{architecture}_{mode}'
  if key in ids:
    sweep_id = ids[key]
    try:
      api = wandb.Api()
      api.sweep(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{sweep_id}')
      print(f'Reusing existing sweep: {sweep_id}')
      return sweep_id
    except Exception:
      print(f'Cached sweep {sweep_id} no longer exists on WandB; creating a new one.')
  sweep_id = wandb.sweep(SWEEP_CONFIGS[(architecture, mode)], entity=WANDB_ENTITY, project=WANDB_PROJECT)
  save_sweep_id(architecture, mode, sweep_id)
  print(f'Created new sweep: {sweep_id}')
  return sweep_id


def count_existing_runs(sweep_id):
  """
  Counts ALL runs regardless of state (finished, crashed, failed, or still
  running): each represents a trial the sweep controller already
  dispatched, including stale entries from a hard disconnect. Excluding
  them would let the effective trial budget silently grow past 30.
  """
  api = wandb.Api()
  sweep = api.sweep(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{sweep_id}')
  return len(sweep.runs)


def launch_sweep(architecture, mode, total_trials=NUM_TRIALS_PER_SWEEP):
  sweep_id = get_or_create_sweep_id(architecture, mode)
  already_run = count_existing_runs(sweep_id)
  remaining = total_trials - already_run

  print(f'{architecture}/{mode}: {already_run}/{total_trials} trials already logged.')

  if remaining <= 0:
    print('Sweep already complete. Nothing to launch.')
    return

  print(f'Launching {remaining} more trial(s)...')
  wandb.agent(sweep_id, function=make_sweep_trial_fn(architecture, mode), count=remaining)

#### ResNet50 TL and from-scratch runs

In [ ]:
launch_sweep('resnet50', 'TL')

In [ ]:
launch_sweep('resnet50', 'scratch')

#### EfficientNet-B0 TL and from-scratch runs

In [ ]:
launch_sweep('efficientnet_b0', 'TL')

In [ ]:
launch_sweep('efficientnet_b0', 'scratch')

#### Swin Tiny TL and from-scratch runs

In [ ]:
launch_sweep('swin_tiny', 'TL')

In [ ]:
launch_sweep('swin_tiny', 'scratch')

### 6.5 Sweep Winner Selection

Lexicographic selection: among the top 15% of trials by macro-F2 per sweep (rank-based), the winner is normally the trial with the smallest 3-epoch-smoothed, fold-averaged train/val loss gap. That override is only trusted if the stability improvement clears 15% of the pool's own gap spread; otherwise the top-F2 trial wins by default, since a smaller improvement is more likely fold-to-fold noise than a real generalisation signal. Applied uniformly across all six sweeps; results persisted to `sweep_winners.json`.

In [ ]:
def get_trial_records(sweep_id):
  """
  Pulls every successfully finished trial from a sweep: config, avg_val_f2,
  and a per-trial stability gap; the mean, across 5 folds, of each fold's
  (3-epoch-smoothed val_loss - 3-epoch-smoothed train_loss) at that fold's
  own stopping point. Small/negative gap = healthy; large positive = overfit
  signature.
  """
  api = wandb.Api()
  sweep = api.sweep(f'{WANDB_ENTITY}/{WANDB_PROJECT}/{sweep_id}')

  records = []
  for run in sweep.runs:
    if run.state != 'finished':
      continue

    summary = run.summary
    if 'avg_val_f2' not in summary:
      continue

    history_df = pd.DataFrame(run.scan_history())
    if '_step' in history_df.columns:
      history_df = history_df.sort_values('_step')

    fold_gaps = []
    for fold in range(5):
      stop_epoch = summary.get(f'fold_{fold}/final_stopping_epoch')
      if stop_epoch is None:
        continue

      fold_rows = history_df[history_df[f'fold_{fold}/epoch'].notna()]
      window = fold_rows[fold_rows[f'fold_{fold}/epoch'] <= stop_epoch].tail(3)

      smoothed_train = window[f'fold_{fold}/train_loss'].mean()
      smoothed_val = window[f'fold_{fold}/val_loss'].mean()
      fold_gaps.append(smoothed_val - smoothed_train)

    if len(fold_gaps) != 5:
      continue

    records.append({
        'run_id': run.id,
        'config': dict(run.config),
        'avg_val_f2': summary['avg_val_f2'],
        'stability_gap': float(np.mean(fold_gaps)),
        'fold_val_f2': [summary.get(f'fold_{f}/final_val_f2') for f in range(5)],
        'fold_stopping_epoch': [summary.get(f'fold_{f}/final_stopping_epoch') for f in range(5)],
    })

  return pd.DataFrame(records)

In [ ]:
def define_top_pool(trials_df, top_pct=0.15):
  """
  Keeps the top `top_pct` of trials by avg_val_f2, by rank. Chosen over an
  SD-based threshold after checking ResNet50-TL's real distribution: F2 was
  bimodal (head_only trials clustered far below last_1_block/last_2_blocks),
  which inflated an SD-based threshold to include 26 of 30 trials; not a
  real shortlist. Rank-based selection is robust to that distribution shape.
  """
  threshold = trials_df['avg_val_f2'].quantile(1 - top_pct)
  return trials_df[trials_df['avg_val_f2'] >= threshold].sort_values('avg_val_f2', ascending=False)

In [ ]:
def select_sweep_winner(sweep_id, top_pct=0.15, min_relative_gain=0.15):
  """
  Among the top `top_pct` trials by avg_val_f2, prefers the lowest
  stability_gap, but only if that improvement clears min_relative_gain of
  the pool's own gap spread. Otherwise defaults to the top-F2 trial outright.
  """
  trials = get_trial_records(sweep_id)
  pool = define_top_pool(trials, top_pct=top_pct)

  top_f2_row = pool.iloc[0]
  best_gap_row = pool.loc[pool['stability_gap'].idxmin()]

  gap_spread = pool['stability_gap'].max() - pool['stability_gap'].min()
  relative_gain = (top_f2_row['stability_gap'] - best_gap_row['stability_gap']) / gap_spread if gap_spread > 0 else 0

  winner_row = best_gap_row if relative_gain >= min_relative_gain else top_f2_row

  return {
      'run_id': str(winner_row['run_id']),
      'config': winner_row['config'],
      'avg_val_f2': float(winner_row['avg_val_f2']),
      'stability_gap': float(winner_row['stability_gap']),
      'fold_val_f2': [float(x) for x in winner_row['fold_val_f2']],
      'fold_stopping_epoch': [int(x) for x in winner_row['fold_stopping_epoch']],
      'pool_size': int(len(pool)),
      'pool_f2_range': [float(pool['avg_val_f2'].min()), float(pool['avg_val_f2'].max())],
      'selection_reason': 'stability' if winner_row is best_gap_row else 'top_f2_fallback',
  }

In [ ]:
SWEEP_WINNERS_PATH = RESULTS / 'sweep_winners.json'

sweep_winners = {}
for (architecture, mode) in SWEEP_COMBINATIONS:
  key = f'{architecture}_{mode}'
  sweep_id = load_sweep_ids()[key]
  sweep_winners[key] = select_sweep_winner(sweep_id)
  w = sweep_winners[key]
  print(f'{key}: winner={w["run_id"]}, avg_val_f2={w["avg_val_f2"]:.4f}, '
        f'stability_gap={w["stability_gap"]:.4f}, reason={w["selection_reason"]}')

with open(SWEEP_WINNERS_PATH, 'w') as f:
  json.dump(sweep_winners, f, indent=2)
print(f'\nSaved: {SWEEP_WINNERS_PATH}')

#### Audit Check: top 15% per sweep

In [ ]:
for (architecture, mode) in SWEEP_COMBINATIONS:
  key = f'{architecture}_{mode}'
  sweep_id = load_sweep_ids()[key]
  trials = get_trial_records(sweep_id)
  pool = define_top_pool(trials, top_pct=0.15)

  print(f'=== {key} ===')
  print(f"F2 range (all 30): {trials['avg_val_f2'].min():.4f} to {trials['avg_val_f2'].max():.4f}")
  print(pool[['run_id', 'avg_val_f2', 'stability_gap']].to_string(index=False))
  winner_id = sweep_winners[key]['run_id']
  print(f'--> Selected winner: {winner_id}\n')

## 7. Arm-Aware 5-Fold Cross-Validation

Per-arm CV results for each of the six winning configs. BUSI_annotated and BUSI_masked genuinely need fresh 5-fold CV, since the sweep never touched their pixel content.

### 7.1 Inpainted: Reuse Sweep Data

In [ ]:
CV_RESULTS_PATH = RESULTS / 'cv_results.json'

def load_cv_results():
  if CV_RESULTS_PATH.exists():
    with open(CV_RESULTS_PATH) as f:
      return json.load(f)
  return {}

def save_cv_results(cv_results):
  with open(CV_RESULTS_PATH, 'w') as f:
    json.dump(cv_results, f, indent=2)

In [ ]:
def get_inpainted_cv_results(winner):
  """
  Reformats a sweep winner's already-completed 5 fold results into the same
  shape run_kfold_cv() produces.
  """
  fold_results = [
      {'fold': i, 'val_f2': winner['fold_val_f2'][i], 'stopping_epoch': winner['fold_stopping_epoch'][i]}
      for i in range(5)
  ]
  return {'fold_results': fold_results, 'avg_val_f2': winner['avg_val_f2'], 'source': 'sweep_reuse'}

In [ ]:
cv_results = load_cv_results()

for (architecture, mode) in SWEEP_COMBINATIONS:
  key = f'{architecture}_{mode}_inpainted'
  if key not in cv_results:
    winner = sweep_winners[f'{architecture}_{mode}']
    cv_results[key] = get_inpainted_cv_results(winner)

save_cv_results(cv_results)

inpainted_keys = sorted(k for k in cv_results if k.endswith('_inpainted'))
print(f'Inpainted CV results: {len(inpainted_keys)}/6 present')
for k in inpainted_keys:
  v = cv_results[k]
  epochs = [r['stopping_epoch'] for r in v['fold_results']]
  print(f'{k}: avg_val_f2={v["avg_val_f2"]:.4f}, fold_stopping_epochs={epochs}')

### 7.2 Annotated & Masked: Fresh 5-Fold CV

Real training: 6 winning configs × 2 untouched arms = 12 CV runs, 5 folds each, using each winner's locked hyperparameters via `run_kfold_cv (context='final')`. Saved after each combination completes, as a precaution against the possible disconnections. Individual folds are separately crash-protected too, via `train_model`'s existing resume mechanism.

In [ ]:
FRESH_ARMS = ['annotated', 'masked']

cv_results = load_cv_results()

for (architecture, mode) in SWEEP_COMBINATIONS:
  winner = sweep_winners[f'{architecture}_{mode}']
  config = winner['config']

  for arm in FRESH_ARMS:
    key = f'{architecture}_{mode}_{arm}'

    if key in cv_results:
      print(f'{key}: already complete, skipping.')
      continue

    print(f'{key}: starting fresh 5-fold CV...')
    result = run_kfold_cv(
        architecture=architecture, mode=mode, arm=arm,
        dropout=config['dropout'], lr=config['lr'], weight_decay=config['weight_decay'],
        epochs=config['epochs'], patience=config['patience'], grace_period=config['grace_period'],
        unfreeze_category=config.get('unfreeze_category'), context='final',
    )
    result['source'] = 'fresh_cv'

    cv_results[key] = result
    save_cv_results(cv_results)
    print(f'{key}: complete -- avg_val_f2={result["avg_val_f2"]:.4f}')

print('\nAll 12 annotated/masked CV runs complete.')

## 8. Final Model Retrain

For each of the 18 architecture × mode × arm combinations: retraining on full 85% CV pool (no validation split) under the winning sweep's locked hyperparameters, for exactly the median of that combination's 5 CV fold stopping epochs. No early stopping; the CV folds already determined when to stop, and this run just applies that number directly.

In [ ]:
FINAL_RETRAIN_ARMS = ['inpainted', 'annotated', 'masked']
FINAL_MODELS_PATH = RESULTS / 'final_models.json'

def load_final_models():
  if FINAL_MODELS_PATH.exists():
    with open(FINAL_MODELS_PATH) as f:
      return json.load(f)
  return {}

def save_final_models(final_models):
  with open(FINAL_MODELS_PATH, 'w') as f:
    json.dump(final_models, f, indent=2)

def compute_median_stopping_epoch(cv_result):
  epochs = [r['stopping_epoch'] for r in cv_result['fold_results']]
  return int(np.median(epochs))


def retrain_final_model(architecture, mode, arm, config, target_epochs, checkpoint_dir, resume=True):
  seed_everything(SEED)

  checkpoint_dir = Path(checkpoint_dir)
  checkpoint_dir.mkdir(parents=True, exist_ok=True)
  latest_path = checkpoint_dir / 'latest.pt'
  final_path = checkpoint_dir / 'final.pt'

  if final_path.exists():
    print(f'{architecture}/{mode}/{arm}: already retrained, skipping.')
    return {'checkpoint_path': str(final_path), 'skipped': True}

  base_dir = LOCAL_DATA_DIR / 'BUSI' / f'BUSI_{arm}'
  mean, std = (IMAGENET_MEAN, IMAGENET_STD) if mode == 'TL' else compute_dataset_stats(busi_cv_85, base_dir)

  train_ds = BreastUltrasoundDataset(busi_cv_85, base_dir, build_transform('train', mean, std))
  train_loader = build_dataloader(train_ds, train=True)
  class_weights = compute_class_weights(busi_cv_85, malignant_boost=MALIGNANT_BOOST)

  model = build_model(architecture, mode, config['dropout'], unfreeze_category=config.get('unfreeze_category')).to(device)
  optimizer = build_optimizer(model, architecture, mode, config['lr'], config['weight_decay'])
  scheduler = build_scheduler(optimizer, architecture, mode, target_epochs)
  scaler = GradScaler('cuda')
  criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

  # Inert; exists only so save/load_latest_checkpoint's existing signature
  # is satisfied. Cannot actually trigger: patience/grace_period exceed
  # target_epochs, and epoch count is controlled directly by the loop below.
  stopper = EarlyStopper(patience=target_epochs + 1, grace_period=target_epochs + 1)

  identity = build_run_identity(architecture, mode, arm, job_type='final_retrain')
  wandb_config = {**config, 'target_epochs': target_epochs}

  start_epoch = 0
  wandb_run_id = None
  if resume and latest_path.exists():
    start_epoch, wandb_run_id = load_latest_checkpoint(latest_path, model, optimizer, scheduler, scaler, stopper)
    print(f'Resumed from checkpoint at epoch {start_epoch}')

  if wandb_run_id:
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, id=wandb_run_id, resume='must',
                     settings=wandb.Settings(silent=True), **identity)
  else:
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, config=wandb_config, resume='allow',
                     settings=wandb.Settings(silent=True), **identity)

  for epoch in range(start_epoch, target_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    if scheduler:
      scheduler.step()
    save_latest_checkpoint(latest_path, model, optimizer, scheduler, epoch, stopper, scaler, wandb_run_id=run.id)
    run.log({'epoch': epoch, 'train_loss': train_loss})

  save_best_checkpoint(final_path, model)
  latest_path.unlink()
  run.finish()

  print(f'{architecture}/{mode}/{arm}: retrain complete -- {target_epochs} epochs')
  return {'checkpoint_path': str(final_path), 'skipped': False}

In [ ]:
sweep_winners = json.load(open(SWEEP_WINNERS_PATH))
cv_results = load_cv_results()
final_models = load_final_models()

for (architecture, mode) in SWEEP_COMBINATIONS:
  config = sweep_winners[f'{architecture}_{mode}']['config']
  for arm in FINAL_RETRAIN_ARMS:
    key = f'{architecture}_{mode}_{arm}'
    if key in final_models:
      print(f'{key}: already recorded, skipping.')
      continue

    target_epochs = compute_median_stopping_epoch(cv_results[key])
    checkpoint_dir = CHECKPOINTS / CHECKPOINT_ARCH_NAMES[architecture] / CHECKPOINT_MODE_NAMES[mode] / f'{arm}_final_retrain'

    result = retrain_final_model(architecture, mode, arm, config, target_epochs, checkpoint_dir)
    final_models[key] = {'architecture': architecture, 'mode': mode, 'arm': arm,
                          'target_epochs': target_epochs, 'checkpoint_path': result['checkpoint_path']}
    save_final_models(final_models)

print(f'\n{len(final_models)}/18 final models recorded.')

### 8.1 Consolidated Results Export

Merges sweep_winners.json, cv_results.json, and final_models.json into one row per architecture × mode × arm combination (18 rows): winning hyperparameters, this arm's CV performance, final retrain length, and checkpoint path. Exported as the single file Notebook 04 reads from, rather than reconstructing this join itself.

In [ ]:
def build_results_summary():
  sweep_winners = json.load(open(SWEEP_WINNERS_PATH))
  cv_results = load_cv_results()
  final_models = load_final_models()

  rows = []
  for (architecture, mode) in SWEEP_COMBINATIONS:
    winner = sweep_winners[f'{architecture}_{mode}']
    config = winner['config']

    for arm in FINAL_RETRAIN_ARMS:
      key = f'{architecture}_{mode}_{arm}'
      cv = cv_results[key]
      fm = final_models[key]

      rows.append({
          'architecture': architecture, 'mode': mode, 'arm': arm,
          'dropout': config['dropout'], 'lr': config['lr'], 'weight_decay': config['weight_decay'],
          'unfreeze_category': config.get('unfreeze_category'),
          'sweep_avg_val_f2': winner['avg_val_f2'],
          'sweep_stability_gap': winner['stability_gap'],
          'sweep_selection_reason': winner['selection_reason'],
          'cv_avg_val_f2': cv['avg_val_f2'],
          'cv_source': cv['source'],
          'cv_fold_stopping_epochs': cv['fold_results'] and [r['stopping_epoch'] for r in cv['fold_results']],
          'final_retrain_epochs': fm['target_epochs'],
          'final_checkpoint_path': fm['checkpoint_path'],
      })

  return pd.DataFrame(rows)


results_summary = build_results_summary()
summary_path = RESULTS / 'notebook03_results_summary.csv'
results_summary.to_csv(summary_path, index=False)

print(f'Exported: {summary_path}')
print(f'Shape: {results_summary.shape} (expected: (18, 15))')
results_summary[['architecture', 'mode', 'arm', 'cv_avg_val_f2', 'final_retrain_epochs']]

In [ ]:
missing = [p for p in results_summary['final_checkpoint_path'] if not Path(p).exists()]
print(f'Missing checkpoint files: {len(missing)} (expected: 0)')
if missing:
  print(missing)

### 8.2 Computational Cost Reporting

Parameter counts (total/trainable, per model), inference FLOPs (per architecture; forward-pass cost is identical regardless of mode/arm/freezing), wall-clock training time (pulled from WandB), and checkpoint size on disk.

In [ ]:
!pip install fvcore --quiet

from fvcore.nn import FlopCountAnalysis

def compute_inference_macs(architecture):
  """
  Forward-pass MACs only; structurally identical regardless of mode,
  arm, dropout, or unfreeze_category, so this is computed once per
  architecture, not once per row. `fvcore` reports MACs, not true FLOPs
  (confirmed against `fvcore`'s own documentation); both are returned,
  labeled explicitly, since papers in this space are inconsistent about
  which convention they report.
  """
  model = build_model(architecture, mode='TL', dropout=0.3, unfreeze_category='head_only')
  model.eval()
  dummy_input = torch.randn(1, 3, 224, 224)

  flops = FlopCountAnalysis(model, dummy_input)
  flops.unsupported_ops_warnings(False)  # informational only -- see note below
  macs = flops.total()
  return {'macs': macs, 'flops_2x_convention': macs * 2}


inference_cost = {arch: compute_inference_macs(arch) for arch in ARCHITECTURE_BUILDERS}
for arch, cost in inference_cost.items():
  print(f"{arch}: {cost['macs']/1e9:.3f} GMACs / {cost['flops_2x_convention']/1e9:.3f} GFLOPs (2x convention)")

In [ ]:
def get_param_counts(architecture, mode, dropout, unfreeze_category):
  model = build_model(architecture, mode, dropout, unfreeze_category=unfreeze_category)
  total = sum(p.numel() for p in model.parameters())
  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  return total, trainable


def get_training_time_seconds(architecture, mode, arm):
  """
  Wall-clock time, pulled from WandB's automatically-tracked run duration --
  never explicitly logged by us, so this recovers it retroactively rather
  than requiring a re-run, same pattern as the stability-gap pull in 6.5.
  """
  api = wandb.Api()
  group = f'{architecture}_{mode}_{arm}'
  runs = api.runs(f'{WANDB_ENTITY}/{WANDB_PROJECT}', filters={'group': group})
  matching = [r for r in runs if r.job_type == 'final_retrain']
  if len(matching) != 1:
    raise ValueError(f'Expected exactly 1 final_retrain run for {group}, found {len(matching)}')
  return matching[0].summary.get('_runtime')


cost_rows = []
for _, row in results_summary.iterrows():
  total_params, trainable_params = get_param_counts(row['architecture'], row['mode'], row['dropout'], row['unfreeze_category'])
  training_seconds = get_training_time_seconds(row['architecture'], row['mode'], row['arm'])
  model_size_mb = Path(row['final_checkpoint_path']).stat().st_size / (1024 ** 2)
  arch_cost = inference_cost[row['architecture']]

  cost_rows.append({
      'architecture': row['architecture'], 'mode': row['mode'], 'arm': row['arm'],
      'total_params': total_params, 'trainable_params': trainable_params,
      'trainable_pct': round(100 * trainable_params / total_params, 2),
      'inference_gmacs': round(arch_cost['macs'] / 1e9, 3),
      'inference_gflops_2x': round(arch_cost['flops_2x_convention'] / 1e9, 3),
      'training_time_min': round(training_seconds / 60, 1) if training_seconds else None,
      'model_size_mb': round(model_size_mb, 2),
  })

cost_df = pd.DataFrame(cost_rows)
results_summary_full = results_summary.merge(cost_df, on=['architecture', 'mode', 'arm'])

cost_path = RESULTS / 'notebook03_results_summary.csv'
results_summary_full.to_csv(cost_path, index=False)

print(f'Updated export: {cost_path}')
print(f'Shape: {results_summary_full.shape} (expected: (18, 22))')
cost_df